# Experiment 3 — XGBoost + SMOTE

Creates synthetic signal events by interpolating between existing ones.

**CRITICAL RULES:**
- SMOTE applied ONLY to X_train — NEVER to test set (data leakage)
- Version A cap: 500k rows (CPU/RAM constraint) — noted as paper limitation
- Versions B and C: NO cap — use full training set (already small due to downsampling)

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

PARAMS = dict(n_estimators=500, max_depth=6, learning_rate=0.1,
              random_state=42, eval_metric='auc', verbosity=0,
              use_label_encoder=False)

# Cap applies ONLY to Version A — B and C use full training data
SMOTE_CAP_VERSION_A = 500_000
print(f"Experiment 3 — XGBoost + SMOTE | Version A cap: {SMOTE_CAP_VERSION_A:,} rows")

Experiment 3 — XGBoost + SMOTE | Version A cap: 500,000 rows


In [2]:
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp3-XGB] Processing Version {v}...")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train = train.drop('label',axis=1).values; y_train = train['label'].values
    X_test  = test.drop('label',axis=1).values;  y_test  = test['label'].values

    # Cap applies ONLY to Version A — B and C use full training data
    if v == 'A' and len(X_train) > SMOTE_CAP_VERSION_A:
        print(f"[Exp3-XGB] Version A: capping to {SMOTE_CAP_VERSION_A:,} rows for SMOTE.")
        idx     = np.random.RandomState(42).choice(len(X_train), SMOTE_CAP_VERSION_A, replace=False)
        X_smote = X_train[idx]; y_smote = y_train[idx]
    else:
        print(f"[Exp3-XGB] Version {v}: full training set ({len(X_train):,} rows) for SMOTE.")
        X_smote = X_train; y_smote = y_train

    smote   = SMOTE(random_state=42)
    t_smote = time.time()
    X_res, y_res = smote.fit_resample(X_smote, y_smote)
    smote_time   = time.time() - t_smote
    print(f"[Exp3-XGB] SMOTE {smote_time:.1f}s | {len(X_res):,} rows | Sig={y_res.sum():,} | BG={(y_res==0).sum():,}")

    model = XGBClassifier(**PARAMS)
    t0    = time.time()
    model.fit(X_res, y_res)
    train_time = time.time() - t0 + smote_time

    probs = model.predict_proba(X_test)[:,1]
    preds = model.predict(X_test)

    metrics = compute_all_metrics(y_test, probs, preds, train_time)
    metrics['Version'] = v
    all_results.append(metrics)

    np.save(os.path.join(RESULTS_DIR, f'probs_exp3_xgb_version_{v}.npy'), probs)
    print_metrics(metrics, label=f"Exp3-XGB Version {v}")

print("\n[Exp3-XGB] Complete.")


[Exp3-XGB] Processing Version A...
[Exp3-XGB] Version A: full training set (400,000 rows) for SMOTE.
[Exp3-XGB] SMOTE 73.6s | 423,524 rows | Sig=211,762 | BG=211,762
[Exp3-XGB Version A] AUC=0.8211 | F1=0.7508 | Signal_Sig=178.8780 | Train_Time=734.38s

[Exp3-XGB] Processing Version B...
[Exp3-XGB] Version B: full training set (207,060 rows) for SMOTE.
[Exp3-XGB] SMOTE 3.1s | 376,474 rows | Sig=188,237 | BG=188,237
[Exp3-XGB Version B] AUC=0.7610 | F1=0.3404 | Signal_Sig=24.5468 | Train_Time=630.65s

[Exp3-XGB] Processing Version C...
[Exp3-XGB] Version C: full training set (192,001 rows) for SMOTE.
[Exp3-XGB] SMOTE 2.1s | 376,474 rows | Sig=188,237 | BG=188,237
[Exp3-XGB Version C] AUC=0.7094 | F1=0.1162 | Signal_Sig=4.6263 | Train_Time=446.96s

[Exp3-XGB] Complete.


In [3]:
results_df = pd.DataFrame(all_results)[['Version','AUC','F1','Signal_Significance','Train_Time_sec']]
save_path  = os.path.join(RESULTS_DIR, 'experiment3_smote_xgb.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      AUC       F1  Signal_Significance  Train_Time_sec
      A 0.821135 0.750783           178.878019          734.38
      B 0.761019 0.340442            24.546807          630.65
      C 0.709420 0.116152             4.626338          446.96

✓ Saved → ..\results\experiment3_smote_xgb.csv
